# 05 - Issues and Editorial Positioning

                This notebook treats issues as their own analytical unit and adds editorial-role information where manually supplied. It phrases special issues as curatorial concentrations, not causal interventions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
article_topics = pd.read_csv(ROOT / "outputs/tables/article_topics.csv")
article_persons = pd.read_csv(ROOT / "outputs/tables/article_persons.csv")
persons = pd.read_csv(ROOT / "outputs/tables/persons_clean.csv")
embedding_index = pd.read_csv(ROOT / "outputs/tables/article_embedding_index.csv")
embeddings = np.load(ROOT / "outputs/models/article_embeddings_main.npy")
roles = pd.read_csv(ROOT / "data/editors_roles.csv")

topic_articles = articles.merge(article_topics[["article_id", "topic_id", "topic_label_human", "topic_macro_category"]], on="article_id", how="left")
embedding_lookup = dict(zip(embedding_index["article_id"], embedding_index["embedding_row"]))
first_author_year = article_persons.groupby("person_id")["year"].min().to_dict()
editor_names = set(roles["person_name"].map(compact_ws)) if not roles.empty else set()

## Issue-Level Metrics

In [ ]:
issue_rows = []
for issue_id, group in topic_articles.groupby("issue_id"):
    topic_counts = group["topic_id"].value_counts(dropna=False)
    topic_probs = topic_counts / topic_counts.sum()
    hhi = float((topic_probs ** 2).sum())
    main_topic = topic_counts.index[0] if len(topic_counts) else pd.NA
    main_topic_row = group[group["topic_id"].eq(main_topic)].head(1)
    main_label = main_topic_row["topic_label_human"].iloc[0] if len(main_topic_row) else pd.NA
    main_macro = main_topic_row["topic_macro_category"].iloc[0] if len(main_topic_row) else pd.NA

    rows = [embedding_lookup.get(aid) for aid in group["article_id"] if aid in embedding_lookup]
    if len(rows) > 1:
        mat = embeddings[rows]
        centroid = mat.mean(axis=0, keepdims=True)
        coherence = float(cosine_similarity(mat, centroid).mean())
    else:
        coherence = np.nan

    issue_persons = article_persons[article_persons["article_id"].isin(group["article_id"])]
    issue_year = int(group["year"].median())
    unique_persons = issue_persons["person_id"].dropna().unique()
    if len(unique_persons):
        new_share = np.mean([first_author_year.get(pid, issue_year) >= issue_year for pid in unique_persons])
        recurring_share = np.mean([first_author_year.get(pid, issue_year) < issue_year for pid in unique_persons])
    else:
        new_share = np.nan
        recurring_share = np.nan

    editor_authored_articles = set(issue_persons.loc[issue_persons["person_name"].isin(editor_names), "article_id"])
    share_editor_authored = len(editor_authored_articles) / group["article_id"].nunique() if group["article_id"].nunique() else np.nan
    issue_title = compact_ws(group["issue_title"].dropna().iloc[0]) if group["issue_title"].notna().any() else ""
    enough_articles_for_concentration = group["article_id"].nunique() >= 4
    special_likely = bool(issue_title) or (enough_articles_for_concentration and hhi >= 0.45)
    basis = "issue_title" if issue_title else "topic_concentration" if enough_articles_for_concentration and hhi >= 0.45 else "none"

    issue_rows.append(
        {
            "issue_id": issue_id,
            "year": issue_year,
            "volume": group["volume"].dropna().iloc[0] if group["volume"].notna().any() else pd.NA,
            "issue": group["issue"].dropna().iloc[0] if group["issue"].notna().any() else pd.NA,
            "journal": group["journal"].dropna().iloc[0] if group["journal"].notna().any() else pd.NA,
            "issue_title": issue_title,
            "n_articles": group["article_id"].nunique(),
            "n_core_articles": int(group["corpus_inclusion_core"].sum()),
            "main_topic": main_label,
            "main_topic_id": main_topic,
            "main_macro_category": main_macro,
            "topic_concentration": hhi,
            "semantic_coherence": coherence,
            "share_editorial_or_intro": float(group["is_editorial_or_intro"].mean()),
            "share_new_authors": new_share,
            "share_recurring_authors": recurring_share,
            "share_editor_authored": share_editor_authored,
            "is_special_issue_likely": special_likely,
            "issue_detection_basis": basis,
            "manual_issue_status": "unclear",
        }
    )

issues = pd.DataFrame(issue_rows).sort_values(["year", "volume", "issue"])
corrections_path = ROOT / "data/issue_corrections.csv"
corrections = pd.read_csv(corrections_path)
if not corrections.empty:
    issues = issues.merge(corrections, on="issue_id", how="left", suffixes=("", "_manual"))
    issues["manual_issue_status"] = np.where(issues["manual_issue_status_manual"].map(compact_ws) != "", issues["manual_issue_status_manual"], issues["manual_issue_status"])
    issues["issue_title"] = np.where(issues["issue_title_corrected"].map(compact_ws) != "", issues["issue_title_corrected"], issues["issue_title"])
    issues = issues.drop(columns=[c for c in issues.columns if c.endswith("_manual") or c == "issue_title_corrected"], errors="ignore")

write_csv(issues, ROOT / "outputs/tables/issues_clean.csv")
display(issues.sort_values(["is_special_issue_likely", "topic_concentration"], ascending=False).head(20))

## Special Issue Pre/Post Patterns

In [ ]:
prepost_rows = []
for row in issues[issues["is_special_issue_likely"]].itertuples():
    if pd.isna(row.main_topic_id):
        continue
    pre = topic_articles[(topic_articles["year"].between(row.year - 5, row.year - 1))]
    issue_year = topic_articles[topic_articles["year"].eq(row.year)]
    post = topic_articles[(topic_articles["year"].between(row.year + 1, row.year + 5))]
    def share(df):
        return float(df["topic_id"].eq(row.main_topic_id).mean()) if len(df) else np.nan
    prepost_rows.append(
        {
            "issue_id": row.issue_id,
            "year": row.year,
            "issue_title": row.issue_title,
            "main_topic": row.main_topic,
            "topic_share_pre_5y": share(pre),
            "topic_share_issue_year": share(issue_year),
            "topic_share_post_5y": share(post),
            "persistence_after_issue": share(post),
            "interpretation_guardrail": "curatorial concentration, not causal proof",
        }
    )
prepost = pd.DataFrame(prepost_rows)
write_csv(prepost, ROOT / "outputs/tables/special_issue_topic_prepost.csv")

fig, ax = plt.subplots(figsize=(12, 5))
plot_issues = issues[issues["is_special_issue_likely"]].copy()
ax.scatter(plot_issues["year"], plot_issues["topic_concentration"], s=plot_issues["n_articles"] * 15, c=plot_issues["semantic_coherence"], cmap="viridis", alpha=0.75)
ax.set_title("Editorial interventions timeline")
ax.set_xlabel("Year")
ax.set_ylabel("Issue topic concentration")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_09_editorial_interventions_timeline.png", dpi=220)
save_caption(ROOT, "fig_09_editorial_interventions_timeline.png", "Likely special issues and topic concentration over time; the figure indicates curatorial concentration, not causal effects.")
plt.show()
display(prepost.head(20))

## Editor Topic Positions

In [ ]:
if roles.empty:
    editor_positions = pd.DataFrame(columns=["person_id", "person_name", "n_articles", "dominant_topic", "topic_entropy_norm", "editor_role_certainty"])
    editor_matched = pd.DataFrame(columns=["topic_id", "topic_label_human", "editor_share", "matched_noneditor_share"])
    editor_ranking = pd.DataFrame()
    write_csv(editor_positions, ROOT / "outputs/tables/editor_topic_positions.csv")
    write_csv(editor_matched, ROOT / "outputs/tables/editor_vs_matched_noneditor_topic_distribution.csv")
    write_csv(editor_ranking, ROOT / "outputs/tables/editor_brokerage_ranking.csv")
    print("No editor roles supplied yet. Fill data/editors_roles.csv and rerun this notebook.")
else:
    role_names = set(roles["person_name"].map(compact_ws))
    editor_persons = article_persons[article_persons["person_name"].isin(role_names)]
    editor_articles = editor_persons.merge(topic_articles[["article_id", "topic_id", "topic_label_human", "year", "language", "type_en", "text_length_words"]], on="article_id", how="left")
    rows = []
    for person_name, group in editor_articles.groupby("person_name"):
        topic_entropy = normalized_entropy(group["topic_id"], n_categories=topic_articles["topic_id"].nunique())
        dominant = group["topic_label_human"].value_counts().index[0] if group["topic_label_human"].notna().any() else pd.NA
        rows.append(
            {
                "person_name": person_name,
                "n_articles": group["article_id"].nunique(),
                "first_year": group["year"].min(),
                "last_year": group["year"].max(),
                "dominant_topic": dominant,
                "n_topics": group["topic_id"].nunique(),
                "topic_entropy_norm": topic_entropy,
            }
        )
    editor_positions = pd.DataFrame(rows)
    if not editor_positions.empty:
        editor_positions = editor_positions.merge(persons, on="person_name", how="left", suffixes=("", "_person"))
    write_csv(editor_positions, ROOT / "outputs/tables/editor_topic_positions.csv")

    if editor_articles.empty:
        editor_matched = pd.DataFrame(columns=["topic_id", "topic_label_human", "n_editor_articles", "editor_share", "n_noneditor_articles", "matched_noneditor_share"])
    else:
        topic_dist_editor = editor_articles.groupby(["topic_id", "topic_label_human"]).size().reset_index(name="n_editor_articles")
        topic_dist_editor["editor_share"] = topic_dist_editor["n_editor_articles"] / max(topic_dist_editor["n_editor_articles"].sum(), 1)
        non_editor_articles = topic_articles[~topic_articles["article_id"].isin(editor_articles["article_id"])]
        topic_dist_non = non_editor_articles.groupby(["topic_id", "topic_label_human"]).size().reset_index(name="n_noneditor_articles")
        topic_dist_non["matched_noneditor_share"] = topic_dist_non["n_noneditor_articles"] / max(topic_dist_non["n_noneditor_articles"].sum(), 1)
        editor_matched = topic_dist_editor.merge(topic_dist_non, on=["topic_id", "topic_label_human"], how="outer").fillna(0)
    write_csv(editor_matched, ROOT / "outputs/tables/editor_vs_matched_noneditor_topic_distribution.csv")

    editor_ranking = editor_positions.sort_values(["topic_entropy_norm", "n_articles"], ascending=False) if not editor_positions.empty else editor_positions
    write_csv(editor_ranking, ROOT / "outputs/tables/editor_brokerage_ranking.csv")

    umap_main = pd.read_csv(ROOT / "outputs/tables/article_umap_main.csv")
    plot_articles = umap_main.merge(topic_articles[["article_id", "topic_label_human"]], on="article_id", how="left")
    editor_article_ids = set(editor_articles["article_id"])
    plot_articles["is_editor_authored"] = plot_articles["article_id"].isin(editor_article_ids)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(plot_articles["umap_x"], plot_articles["umap_y"], s=12, alpha=0.25, color="lightgray")
    highlighted = plot_articles[plot_articles["is_editor_authored"]]
    ax.scatter(highlighted["umap_x"], highlighted["umap_y"], s=28, alpha=0.85, color="#d62728")
    ax.set_title("Editor-authored articles in topic space")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    fig.tight_layout()
    fig.savefig(ROOT / "outputs/figures/fig_14_editor_positions_in_topic_space.png", dpi=220)
    save_caption(ROOT, "fig_14_editor_positions_in_topic_space.png", "Editorial actors overlaid on the exploratory semantic map; this indicates positioning, not causal influence.")
    plt.show()

display(editor_positions.head(20))